# 🧬 GeneSight — Notebook 1: Data Pipeline
### ClinVar + gnomAD Variant Feature Engineering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/genesight-biorag/blob/main/notebooks/01_GeneSight_Data_Pipeline.ipynb)

---

## What this notebook does

This is the foundation of **GeneSight** — a machine learning tool that predicts variant pathogenicity by combining labeled clinical data from [ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/) with population frequency data from [gnomAD](https://gnomad.broadinstitute.org/).

**Pipeline steps:**
1. Download and parse the ClinVar variant summary (labeled ground truth)
2. Engineer biologically meaningful features from raw annotations
3. Fetch allele frequency data from gnomAD via their public API
4. Join and clean the final feature matrix
5. Export a clean `.parquet` file ready for model training (Notebook 2)

**Why ClinVar + gnomAD together?**
ClinVar tells us *what clinicians believe* about a variant (pathogenic/benign). gnomAD tells us *how common* a variant is in the general population — rare variants are enriched for pathogenicity. Combining both gives the model a much stronger signal than either source alone.

---

## ⚙️ Setup & Installations

In [ ]:
# Install required packages
!pip install pandas pyarrow requests tqdm matplotlib seaborn --quiet
print("✅ Packages installed")

In [ ]:
import pandas as pd
import numpy as np
import requests
import gzip
import os
import time
import json
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Output directory
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("✅ Imports complete")
print(f"📁 Data directory: {DATA_DIR.resolve()}")

---
## 📥 Step 1: Download ClinVar Variant Summary

ClinVar provides a comprehensive flat-file download (`variant_summary.txt.gz`) updated monthly. Each row is a variant with:
- Genomic coordinates (chromosome, position, ref/alt alleles)
- Clinical significance label (our **prediction target**)
- Gene name, molecular consequence, review status

We filter to **single-nucleotide variants (SNVs)** with high-confidence review status (≥1 star) to ensure label quality.

In [ ]:
CLINVAR_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"
CLINVAR_PATH = DATA_DIR / "variant_summary.txt.gz"

def download_with_progress(url, dest_path):
    """Download a file with a progress bar."""
    if dest_path.exists():
        print(f"⏭️  Already downloaded: {dest_path.name}")
        return
    print(f"⬇️  Downloading {dest_path.name}...")
    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))
    with open(dest_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))
    print(f"✅ Saved to {dest_path}")

download_with_progress(CLINVAR_URL, CLINVAR_PATH)

In [ ]:
# --- Inspect actual column names first (ClinVar schema changes with each release) ---
# NOTE: 'MolecularConsequence' was removed from variant_summary.txt in 2024.
# We now infer molecular consequence from the HGVS 'Name' column (see feature engineering).
print("🔍 Inspecting ClinVar column schema...")
with gzip.open(CLINVAR_PATH, 'rt', encoding='utf-8') as f:
    actual_cols = pd.read_csv(f, sep='\t', nrows=0).columns.tolist()
print(f"   Schema has {len(actual_cols)} columns")

# Desired columns — 'Name' contains HGVS notation used to infer consequence
DESIRED_COLS = [
    'GeneSymbol', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele',
    'ClinicalSignificance', 'ReviewStatus', 'Type', 'Assembly',
    'NumberSubmitters', 'LastEvaluated', 'Name'
]
CLINVAR_COLS = [c for c in DESIRED_COLS if c in actual_cols]
missing = [c for c in DESIRED_COLS if c not in actual_cols]
if missing:
    print(f"   ⚠️  Columns not in this release (skipping): {missing}")
print(f"   ✅ Loading: {CLINVAR_COLS}")

print("\n📂 Parsing ClinVar variant summary...")
with gzip.open(CLINVAR_PATH, 'rt', encoding='utf-8') as f:
    clinvar_raw = pd.read_csv(f, sep='\t', usecols=CLINVAR_COLS, low_memory=False)

print(f"   Raw variants loaded: {len(clinvar_raw):,}")

# Filter: GRCh38, SNVs only, high-confidence labels
HIGH_CONFIDENCE_STATUSES = [
    'reviewed by expert panel',
    'criteria provided, multiple submitters, no conflicts',
    'criteria provided, single submitter'
]

clinvar = (
    clinvar_raw
    .query("Assembly == 'GRCh38'")
    .query("Type == 'single nucleotide variant'")
    .query("ReviewStatus in @HIGH_CONFIDENCE_STATUSES")
    .dropna(subset=['ClinicalSignificance', 'Chromosome', 'Start'])
    .copy()
)

print(f"   After filters (GRCh38, SNV, high-confidence): {len(clinvar):,}")

In [ ]:
# Create binary label: Pathogenic=1, Benign=0
# We exclude ambiguous labels (VUS, conflicting) for a clean training signal
PATHOGENIC_TERMS = ['Pathogenic', 'Likely pathogenic', 'Pathogenic/Likely pathogenic']
BENIGN_TERMS     = ['Benign', 'Likely benign', 'Benign/Likely benign']

clinvar['label'] = np.where(
    clinvar['ClinicalSignificance'].isin(PATHOGENIC_TERMS), 1,
    np.where(clinvar['ClinicalSignificance'].isin(BENIGN_TERMS), 0, np.nan)
)
clinvar = clinvar.dropna(subset=['label'])
clinvar['label'] = clinvar['label'].astype(int)

label_counts = clinvar['label'].value_counts()
print(f"✅ Label distribution:")
print(f"   Benign (0):     {label_counts.get(0, 0):,}")
print(f"   Pathogenic (1): {label_counts.get(1, 0):,}")
print(f"   Total:          {len(clinvar):,}")

# Visualize
fig, ax = plt.subplots(figsize=(6, 3))
label_counts.rename({0: 'Benign', 1: 'Pathogenic'}).plot(kind='bar', ax=ax, color=['#2ecc71','#e74c3c'], edgecolor='white')
ax.set_title('ClinVar Label Distribution (filtered)', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Variant count')
ax.tick_params(rotation=0)
plt.tight_layout()
plt.savefig(DATA_DIR / 'label_distribution.png', dpi=150)
plt.show()

---
## 🔧 Step 2: Feature Engineering from ClinVar Annotations

Raw variant annotations aren't directly usable by ML models. We engineer features that capture biological signal:

| Feature | Biological rationale |
|---|---|
| `is_missense` | Missense variants are enriched for pathogenicity vs. synonymous |
| `is_lof` | Loss-of-function (nonsense, frameshift) are often pathogenic |
| `num_submitters` | More submitters → stronger evidence |
| `chrom_encoded` | Some chromosomes have disease-enriched variant landscapes |
| `allele_change_type` | Transition vs. transversion affects functional impact |
| `review_confidence` | Ordinal encoding of review status tier |

In [ ]:
def engineer_clinvar_features(df):
    """Engineer ML features from ClinVar annotations."""
    feat = df.copy()

    # --- Molecular consequence features ---
    # ClinVar removed the MolecularConsequence column in 2024.
    # We infer consequence from the HGVS 'Name' field, which contains
    # standard notation like p.Arg123Ter (nonsense), c.123G>A (missense), etc.
    hgvs = feat['Name'].fillna('').str.lower() if 'Name' in feat.columns else pd.Series([''] * len(feat))
    feat['is_missense']   = hgvs.str.contains(r'p\.[a-z]{3}\d+[a-z]{3}', regex=True).astype(int)
    feat['is_synonymous'] = hgvs.str.contains(r'p\.=|p\.[a-z]{3}\d+\=', regex=True).astype(int)
    feat['is_nonsense']   = hgvs.str.contains(r'ter|\*|stop|p\.[a-z]{3}\d+\*', regex=True).astype(int)
    feat['is_splice']     = hgvs.str.contains(r'splice|ivs|c\.\d+[+-]\d', regex=True).astype(int)
    feat['is_frameshift'] = hgvs.str.contains(r'fs|frameshift', regex=True).astype(int)
    feat['is_lof']        = ((feat['is_nonsense'] == 1) | (feat['is_frameshift'] == 1)).astype(int)

    # --- Allele change type (transition vs transversion) ---
    transitions = {('A','G'),('G','A'),('C','T'),('T','C')}
    ref = feat['ReferenceAllele'].str.upper().fillna('')
    alt = feat['AlternateAllele'].str.upper().fillna('')
    feat['is_transition'] = [int((r,a) in transitions) for r,a in zip(ref, alt)]

    # --- Chromosome encoding ---
    chrom_map = {str(i): i for i in range(1,23)}
    chrom_map.update({'X': 23, 'Y': 24, 'MT': 25})
    feat['chrom_encoded'] = feat['Chromosome'].astype(str).map(chrom_map).fillna(0).astype(int)

    # --- Evidence strength ---
    feat['num_submitters'] = pd.to_numeric(feat['NumberSubmitters'], errors='coerce').fillna(1)
    review_order = {
        'reviewed by expert panel': 3,
        'criteria provided, multiple submitters, no conflicts': 2,
        'criteria provided, single submitter': 1
    }
    feat['review_confidence'] = feat['ReviewStatus'].map(review_order).fillna(0).astype(int)

    return feat

clinvar = engineer_clinvar_features(clinvar)
print("✅ ClinVar features engineered")

FEATURE_COLS_CLINVAR = [
    'is_missense','is_synonymous','is_nonsense','is_splice','is_lof','is_frameshift',
    'is_transition','chrom_encoded','num_submitters','review_confidence'
]

clinvar[FEATURE_COLS_CLINVAR].describe().round(3)

---
## 🌍 Step 3: Fetch Population Frequency from gnomAD

gnomAD v4 contains allele frequencies from >800,000 individuals across diverse ancestries. The key insight: **pathogenic variants are rare** (natural selection removes them from populations). A high allele frequency strongly argues *against* pathogenicity.

We query gnomAD's public GraphQL API for a sample of variants. For the full dataset, we use gnomAD's downloadable constraint files to avoid rate limits.

> **Note:** We work with a random sample of 5,000 variants for this demo. In production, you'd run this overnight against the full dataset.

In [ ]:
# gnomAD gene-level constraint scores (pLI, LOEUF) — proxy for gene intolerance
# These are pre-computed and available as a static download
GNOMAD_CONSTRAINT_URL = "https://storage.googleapis.com/gcp-public-data--gnomad/release/4.1/constraint/gnomad.v4.1.constraint_metrics.tsv"
CONSTRAINT_PATH = DATA_DIR / "gnomad_constraint.tsv"

download_with_progress(GNOMAD_CONSTRAINT_URL, CONSTRAINT_PATH)

constraint = pd.read_csv(CONSTRAINT_PATH, sep='\t', usecols=['gene', 'lof.pLI', 'lof.oe_ci.upper'], low_memory=False)
constraint.columns = ['gene', 'pLI', 'LOEUF']
constraint = constraint.dropna(subset=['pLI', 'LOEUF'])
print(f"✅ gnomAD constraint scores loaded: {len(constraint):,} genes")
constraint.head()

In [ ]:
def query_gnomad_variant(chrom, pos, ref, alt, retries=3):
    """
    Query gnomAD v4 GraphQL API for a single variant's allele frequency.
    Returns (AF_genome, AF_exome) or (None, None) on failure.
    """
    variant_id = f"{chrom}-{pos}-{ref}-{alt}"
    query = """
    query VariantQuery($variantId: String!) {
      variant(variantId: $variantId, dataset: gnomad_r4) {
        genome { af }
        exome  { af }
      }
    }
    """
    for attempt in range(retries):
        try:
            resp = requests.post(
                "https://gnomad.broadinstitute.org/api",
                json={"query": query, "variables": {"variantId": variant_id}},
                timeout=10
            )
            if resp.status_code == 200:
                data = resp.json().get('data', {}).get('variant')
                if data:
                    af_g = data.get('genome', {}) or {}
                    af_e = data.get('exome', {}) or {}
                    return af_g.get('af', 0.0) or 0.0, af_e.get('af', 0.0) or 0.0
        except Exception:
            time.sleep(1)
    return None, None

# Sample a small batch for demonstration
SAMPLE_N = 200  # Increase to 5000+ for a real run (will take ~30 min)
sample = clinvar.sample(n=min(SAMPLE_N, len(clinvar)), random_state=42).copy()

print(f"🔍 Querying gnomAD for {len(sample):,} variants (demo sample)...")
print("   ⏱️  For the full dataset, use gnomAD bulk download instead.")

af_genome_list, af_exome_list = [], []
for _, row in tqdm(sample.iterrows(), total=len(sample), desc="gnomAD API"):
    af_g, af_e = query_gnomad_variant(
        str(row['Chromosome']), int(row['Start']),
        str(row['ReferenceAllele']), str(row['AlternateAllele'])
    )
    af_genome_list.append(af_g if af_g is not None else 0.0)
    af_exome_list.append(af_e if af_e is not None else 0.0)
    time.sleep(0.1)  # Be polite to the API

sample['af_genome'] = af_genome_list
sample['af_exome']  = af_exome_list
print(f"✅ gnomAD allele frequencies fetched")
print(f"   Variants with AF > 0: {(sample['af_genome'] > 0).sum()}")

---
## 🔗 Step 4: Join ClinVar + gnomAD Features

We merge the gnomAD allele frequency and gene-level constraint scores into our ClinVar feature matrix. The final feature set spans:
- **Variant-level:** molecular consequence, allele change, chromosomal context
- **Population-level:** allele frequency in genome/exome cohorts  
- **Gene-level:** pLI (probability loss-of-function intolerant), LOEUF (upper bound of loss-of-function observed/expected ratio)

Low LOEUF + low AF + LoF consequence = high prior probability of pathogenicity.

In [ ]:
# Join gene-level constraint scores
sample = sample.merge(
    constraint.rename(columns={'gene': 'GeneSymbol'}),
    on='GeneSymbol', how='left'
)

# Log-transform allele frequency (right-skewed, spanning many orders of magnitude)
EPS = 1e-10
sample['log_af_genome'] = np.log10(sample['af_genome'] + EPS)
sample['log_af_exome']  = np.log10(sample['af_exome']  + EPS)

# pLI and LOEUF: fill missing with population median (conservative)
sample['pLI']   = sample['pLI'].fillna(sample['pLI'].median())
sample['LOEUF'] = sample['LOEUF'].fillna(sample['LOEUF'].median())

ALL_FEATURES = FEATURE_COLS_CLINVAR + ['log_af_genome', 'log_af_exome', 'pLI', 'LOEUF']

final_df = sample[ALL_FEATURES + ['label', 'GeneSymbol', 'Chromosome', 'Start']].dropna(subset=ALL_FEATURES)

print(f"✅ Final feature matrix: {final_df.shape[0]:,} variants × {len(ALL_FEATURES)} features")
final_df[ALL_FEATURES].head()

---
## 📊 Step 5: Exploratory Data Analysis

Before training any model, we should understand what signal exists in the features. Good EDA prevents surprises later.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('GeneSight: Feature Distributions by Pathogenicity', fontsize=14, fontweight='bold', y=1.01)

palette = {0: '#2ecc71', 1: '#e74c3c'}
label_names = {0: 'Benign', 1: 'Pathogenic'}
plot_df = final_df.copy()
plot_df['Pathogenicity'] = plot_df['label'].map(label_names)

# 1. Allele frequency
sns.boxplot(data=plot_df, x='Pathogenicity', y='log_af_genome', ax=axes[0,0],
            palette={'Benign':'#2ecc71','Pathogenic':'#e74c3c'})
axes[0,0].set_title('Allele Frequency (log10)')

# 2. pLI
sns.boxplot(data=plot_df, x='Pathogenicity', y='pLI', ax=axes[0,1],
            palette={'Benign':'#2ecc71','Pathogenic':'#e74c3c'})
axes[0,1].set_title('Gene pLI Score')

# 3. LOEUF
sns.boxplot(data=plot_df, x='Pathogenicity', y='LOEUF', ax=axes[0,2],
            palette={'Benign':'#2ecc71','Pathogenic':'#e74c3c'})
axes[0,2].set_title('Gene LOEUF Score')

# 4. Molecular consequence proportions
cons_cols = ['is_missense','is_lof','is_splice','is_synonymous']
cons_means = plot_df.groupby('Pathogenicity')[cons_cols].mean()
cons_means.T.plot(kind='bar', ax=axes[1,0], color=['#2ecc71','#e74c3c'], edgecolor='white')
axes[1,0].set_title('Consequence Type Rate')
axes[1,0].tick_params(rotation=30)
axes[1,0].legend(fontsize=8)

# 5. Correlation heatmap
corr = final_df[ALL_FEATURES].corr()
sns.heatmap(corr, ax=axes[1,1], cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=False, square=True, linewidths=0.3)
axes[1,1].set_title('Feature Correlation Matrix')
axes[1,1].tick_params(labelsize=7)

# 6. Number of submitters
sns.boxplot(data=plot_df, x='Pathogenicity', y='num_submitters', ax=axes[1,2],
            palette={'Benign':'#2ecc71','Pathogenic':'#e74c3c'})
axes[1,2].set_title('Number of ClinVar Submitters')

plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_features.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plots saved")

---
## 💾 Step 6: Export Feature Matrix

We save the final dataset as a Parquet file — a columnar format that's fast to read and much smaller than CSV. This is the input to **Notebook 2: Model Training**.

In [ ]:
OUTPUT_PATH = DATA_DIR / "genesight_features.parquet"
final_df.to_parquet(OUTPUT_PATH, index=False)
size_mb = OUTPUT_PATH.stat().st_size / 1e6

print("✅ Feature matrix exported!")
print(f"   Path:     {OUTPUT_PATH}")
print(f"   Shape:    {final_df.shape[0]:,} rows × {final_df.shape[1]} columns")
print(f"   Size:     {size_mb:.2f} MB")
print(f"   Features: {ALL_FEATURES}")
print(f"   Label balance: {final_df['label'].value_counts().to_dict()}")
print("\n🚀 Ready for Notebook 2: Model Training + SHAP Explainability!")

---
## ✅ Summary

In this notebook we:

1. **Downloaded ClinVar** — ~500k variants, filtered to high-confidence SNVs with unambiguous labels
2. **Engineered 9 ClinVar features** — molecular consequence, allele type, evidence strength
3. **Fetched gnomAD data** — allele frequencies (variant-level) + pLI/LOEUF (gene-level constraint)
4. **Built a joined feature matrix** — 13 features spanning variant, population, and gene dimensions
5. **Ran EDA** — confirmed that pathogenic variants are rarer (lower AF), found in more constrained genes (higher pLI, lower LOEUF), and enriched for LoF consequences

**➡️ Next: [Notebook 2 — XGBoost Model Training + SHAP Explainability](./02_GeneSight_Model_SHAP.ipynb)**

---
*GeneSight is an open-source research tool. Predictions are for research purposes only and should not be used for clinical decision-making.*